In [0]:
%run ../src/00_PIPELINE_FUNCTIONS

In [0]:
dbutils.widgets.dropdown(
    "environment",
    "DEV",
    ["DEV", "STAGING", "PROD"]
)

dbutils.widgets.dropdown(
    "run_type",
    "INCREMENTAL",
    [
        "INCREMENTAL",
        "BACKFILL",
        "REPROCESS"
    ]
)

dbutils.widgets.dropdown(
    "run_mode",
    "AUTO",
    ["AUTO", "MANUAL"]
)

dbutils.widgets.text(
    "delivery_date",
    ""
)

dbutils.widgets.dropdown(
    "delivery_batch",
    "1600",
    ["0800", "1600", "2300"]
)

In [0]:
from datetime import datetime
from zoneinfo import ZoneInfo


# =========================================================
# 1. LEER PARÁMETROS
# =========================================================

environment = (
    dbutils.widgets.get(
        "environment"
    )
    .strip()
    .upper()
)

run_type = (
    dbutils.widgets.get(
        "run_type"
    )
    .strip()
    .upper()
)

run_mode = (
    dbutils.widgets.get(
        "run_mode"
    )
    .strip()
    .upper()
)


# =========================================================
# 2. CONFIGURAR ENVIRONMENT
# =========================================================

configure_environment(
    environment
)

require_environment()


# =========================================================
# 3. VALIDAR RUN TYPE
# =========================================================

allowed_run_types = [
    "INCREMENTAL",
    "BACKFILL",
    "REPROCESS"
]

if run_type not in allowed_run_types:

    raise ValueError(
        f"INVALID_RUN_TYPE | "
        f"run_type={run_type} | "
        f"allowed={allowed_run_types}"
    )


# =========================================================
# 4. VALIDAR COMBINACIÓN RUN MODE + RUN TYPE
# =========================================================

if (
    run_mode == "AUTO"
    and run_type != "INCREMENTAL"
):

    raise ValueError(
        f"INVALID_RUN_CONFIGURATION | "
        f"run_mode={run_mode} | "
        f"run_type={run_type} | "
        f"AUTO solo permite INCREMENTAL"
    )


# =========================================================
# 5. DERIVAR FORCE REPROCESS
# =========================================================

force_reprocess = (
    run_type == "REPROCESS"
)


# =========================================================
# 6. RESOLVER DELIVERY DATE Y DELIVERY BATCH
# =========================================================

if run_mode == "AUTO":

    now_lima = datetime.now(
        ZoneInfo(
            "America/Lima"
        )
    )

    delivery_date = (
        now_lima
        .date()
        .isoformat()
    )

    current_hour = (
        now_lima.hour
    )

    if current_hour == 8:

        delivery_batch = "0800"

    elif current_hour == 16:

        delivery_batch = "1600"

    elif current_hour == 23:

        delivery_batch = "2300"

    else:

        raise ValueError(
            f"AUTO_RUN_OUTSIDE_SCHEDULE | "
            f"current_time="
            f"{now_lima.strftime('%Y-%m-%d %H:%M:%S')} | "
            f"allowed_hours=[8, 16, 23]"
        )


elif run_mode == "MANUAL":

    delivery_date = (
        dbutils.widgets.get(
            "delivery_date"
        )
        .strip()
    )

    delivery_batch = (
        dbutils.widgets.get(
            "delivery_batch"
        )
        .strip()
    )

    if not delivery_date:

        raise ValueError(
            "delivery_date es obligatorio "
            "cuando run_mode=MANUAL"
        )

    if delivery_batch not in [
        "0800",
        "1600",
        "2300"
    ]:

        raise ValueError(
            f"INVALID_DELIVERY_BATCH | "
            f"delivery_batch={delivery_batch}"
        )


else:

    raise ValueError(
        f"INVALID_RUN_MODE | "
        f"run_mode={run_mode} | "
        f"allowed=['AUTO', 'MANUAL']"
    )


# =========================================================
# 7. CONSTRUIR BATCH ID
# =========================================================

batch_id = (
    delivery_date.replace(
        "-",
        ""
    )
    + "_"
    + delivery_batch
)


# =========================================================
# 8. MOSTRAR CONTEXTO DE EJECUCIÓN
# =========================================================

print(
    f"environment={ENVIRONMENT} | "
    f"catalog={CATALOG} | "
    f"run_type={run_type} | "
    f"run_mode={run_mode} | "
    f"delivery_date={delivery_date} | "
    f"delivery_batch={delivery_batch} | "
    f"batch_id={batch_id} | "
    f"force_reprocess={force_reprocess}"
)

run_mode=MANUAL | delivery_date=2025-01-01 | delivery_batch=1600 | batch_id=20250101_1600 | force_reprocess=False


In [0]:
run_bronze_task(
    delivery_date=delivery_date,
    delivery_batch=delivery_batch,
    force_reprocess=force_reprocess,
    run_type=run_type
)

TASK STARTED | batch=20250101_1600 | stage=BRONZE | task_run_id=dc727e3e-073e-484a-b693-0cd772ef6a9d
Procesando batch: 20250101_1600
Run ID: 90738f87-b124-45aa-9657-f220ca040169
SKIP: 20250101_1600 ya fue procesado correctamente.
TASK FINISHED | task_run_id=dc727e3e-073e-484a-b693-0cd772ef6a9d | status=SUCCESS
BRONZE finalizado correctamente | batch=20250101_1600


In [0]:
dbutils.jobs.taskValues.set(
    key="environment",
    value=ENVIRONMENT
)

dbutils.jobs.taskValues.set(
    key="run_type",
    value=run_type
)

dbutils.jobs.taskValues.set(
    key="force_reprocess",
    value=force_reprocess
)

dbutils.jobs.taskValues.set(
    key="delivery_date",
    value=delivery_date
)

dbutils.jobs.taskValues.set(
    key="delivery_batch",
    value=delivery_batch
)

dbutils.jobs.taskValues.set(
    key="batch_id",
    value=batch_id
)


print(
    f"Task Values publicados | "
    f"environment={ENVIRONMENT} | "
    f"run_type={run_type} | "
    f"force_reprocess={force_reprocess} | "
    f"delivery_date={delivery_date} | "
    f"delivery_batch={delivery_batch} | "
    f"batch_id={batch_id}"
)